# Layer Normalization

## Learning Objectives
1. Understand why normalisation stabilises training and the design choices behind LayerNorm.
2. Implement LayerNorm forward pass from scratch in NumPy with learned gamma/beta.
3. Compare BatchNorm, LayerNorm, GroupNorm, and RMSNorm on a shared benchmark with OOM handling.
4. Analyse pre-norm vs post-norm transformer placement and RMSNorm gradient flow in LLaMA-style models.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


## Level 1: LayerNorm Forward Pass (NumPy)

In [ ]:
# LayerNorm normalises across the feature dimension for each sample independently.
# This makes it batch-size independent — critical for RNNs and transformers
# where batch statistics are unreliable (variable sequence lengths, small batches).

def layer_norm_np(x, gamma, beta, eps=1e-5):
    '''Layer Normalisation over the last axis.
    Args:
        x:     input, shape (batch, features)
        gamma: scale parameter, shape (features,)
        beta:  shift parameter, shape (features,)
        eps:   numerical stability constant
    Returns:
        normalised output, same shape as x
    '''
    mean = x.mean(axis=-1, keepdims=True)       # per-sample mean
    var  = x.var(axis=-1, keepdims=True)        # per-sample variance
    x_hat = (x - mean) / np.sqrt(var + eps)    # normalise
    return gamma * x_hat + beta                 # affine transform


def batch_norm_np(x, gamma, beta, eps=1e-5):
    '''Batch Normalisation over the batch dimension (axis=0).
    Normalises each feature across the batch — batch-size dependent.
    '''
    mean = x.mean(axis=0, keepdims=True)
    var  = x.var(axis=0, keepdims=True)
    x_hat = (x - mean) / np.sqrt(var + eps)
    return gamma * x_hat + beta


# Demonstrate difference in normalisation axes
np.random.seed(42)
B, D = 4, 8   # batch=4, features=8
x    = np.random.randn(B, D) * 3.0 + 1.5
gamma = np.ones(D)
beta  = np.zeros(D)

ln_out = layer_norm_np(x, gamma, beta)
bn_out = batch_norm_np(x, gamma, beta)

print('Input stats per sample:',
      np.round(x.mean(axis=1), 3), '(means)',
      np.round(x.std(axis=1),  3), '(stds)')
print()
print('LayerNorm — each sample has mean~0, std~1 across features:')
print('  means:', np.round(ln_out.mean(axis=1), 5))
print('  stds: ', np.round(ln_out.std(axis=1),  5))
print()
print('BatchNorm — each feature has mean~0, std~1 across batch:')
print('  feature means:', np.round(bn_out.mean(axis=0), 5))
print('  feature stds: ', np.round(bn_out.std(axis=0),  5))

# Verify against PyTorch
ln_torch = nn.LayerNorm(D, elementwise_affine=False)
x_t  = torch.tensor(x, dtype=torch.float32)
diff = (ln_out - ln_torch(x_t).detach().numpy()).max()
print(f'\nMax diff vs torch.nn.LayerNorm: {diff:.2e}  (should be ~1e-6)')


## Level 2: BatchNorm vs LayerNorm vs GroupNorm vs RMSNorm Comparison

In [ ]:
# Compare all four normalisation strategies on a shared training task.
# BatchNorm: best for CNNs with large batches, fails with batch_size=1.
# LayerNorm: standard for transformers and NLP, batch-size independent.
# GroupNorm: intermediate — useful for small-batch image tasks.
# RMSNorm: LayerNorm without mean-centering; faster and used in LLaMA.

import time

torch.manual_seed(0)
D_FEAT = 64
N_CLASSES_NORM = 8

X_norm = torch.randn(1200, D_FEAT, device=device)
y_norm = torch.randint(0, N_CLASSES_NORM, (1200,), device=device)
ds_norm = TensorDataset(X_norm[:960], y_norm[:960])
val_norm = TensorDataset(X_norm[960:], y_norm[960:])
loader_norm = DataLoader(ds_norm, batch_size=32, shuffle=True)
val_loader_norm = DataLoader(val_norm, batch_size=128)


class RMSNorm(nn.Module):
    '''RMS normalisation — omits mean subtraction for speed.
    Used in LLaMA, PaLM, and other modern LLMs.
    '''
    def __init__(self, d, eps=1e-8):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(d))
        self.eps   = eps

    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return self.scale * x / rms


def build_norm_net(norm_type):
    '''Build a 3-layer MLP with the requested normalisation.'''
    layers = [nn.Linear(D_FEAT, 128), nn.ReLU()]
    if norm_type == 'batch':  layers += [nn.BatchNorm1d(128)]
    elif norm_type == 'layer': layers += [nn.LayerNorm(128)]
    elif norm_type == 'group': layers += [nn.GroupNorm(8, 128)]
    elif norm_type == 'rms':   layers += [RMSNorm(128)]
    layers += [nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, N_CLASSES_NORM)]
    return nn.Sequential(*layers).to(device)


results = {}
for norm in ['batch', 'layer', 'group', 'rms']:
    model_n = build_norm_net(norm)
    opt_n   = torch.optim.Adam(model_n.parameters(), lr=1e-3)
    t0      = time.time()
    for _ in range(30):
        model_n.train()
        for xb, yb in loader_norm:
            try:
                opt_n.zero_grad()
                nn.CrossEntropyLoss()(model_n(xb), yb).backward()
                opt_n.step()
            except RuntimeError as exc:
                if 'out of memory' in str(exc).lower():
                    torch.cuda.empty_cache(); continue
                raise
    elapsed = time.time() - t0
    model_n.eval()
    with torch.no_grad():
        correct = sum(
            (model_n(xb).argmax(1) == yb).sum().item()
            for xb, yb in val_loader_norm
        )
    results[norm] = {'acc': correct / len(val_norm), 'time': elapsed}

print(f'{'Norm':<10} {'Val Acc':>8} {'Time (s)':>10}')
print('-' * 32)
for norm, r in results.items():
    print(f'{norm:<10} {r["acc"]:>8.4f} {r["time"]:>10.2f}')

print('\nKey insight: LayerNorm and RMSNorm are batch-size agnostic — '
      'always stable.\nBatchNorm fails with batch_size=1 (common in inference).')


## Real-World Example 1: Pre-Norm vs Post-Norm Transformer

In [ ]:
# Post-Norm (original BERT): residual stream flows through norm.
# Pre-Norm (GPT-2+, LLaMA): normalise BEFORE attention/FFN — more stable training,
# especially at depth; allows higher learning rates without warmup.

torch.manual_seed(10)
D_MODEL = 64
N_HEADS = 4
D_FF    = 256


class TransformerBlockPostNorm(nn.Module):
    '''Post-LN: x = LayerNorm(x + SubLayer(x)) — original Transformer.'''
    def __init__(self):
        super().__init__()
        self.attn  = nn.MultiheadAttention(D_MODEL, N_HEADS, batch_first=True)
        self.ff    = nn.Sequential(
            nn.Linear(D_MODEL, D_FF), nn.GELU(), nn.Linear(D_FF, D_MODEL)
        )
        self.ln1   = nn.LayerNorm(D_MODEL)
        self.ln2   = nn.LayerNorm(D_MODEL)

    def forward(self, x):
        attn_out, _ = self.attn(x, x, x, need_weights=False)
        x = self.ln1(x + attn_out)   # norm AFTER residual addition
        x = self.ln2(x + self.ff(x))
        return x


class TransformerBlockPreNorm(nn.Module):
    '''Pre-LN: x = x + SubLayer(LayerNorm(x)) — modern standard.'''
    def __init__(self):
        super().__init__()
        self.attn  = nn.MultiheadAttention(D_MODEL, N_HEADS, batch_first=True)
        self.ff    = nn.Sequential(
            nn.Linear(D_MODEL, D_FF), nn.GELU(), nn.Linear(D_FF, D_MODEL)
        )
        self.ln1   = nn.LayerNorm(D_MODEL)
        self.ln2   = nn.LayerNorm(D_MODEL)

    def forward(self, x):
        x = x + self.attn(*([self.ln1(x)]*3), need_weights=False)[0]  # norm BEFORE
        x = x + self.ff(self.ln2(x))
        return x


# Train both on sequence classification (1 token per sequence)
torch.manual_seed(11)
X_seq = torch.randn(800, 1, D_MODEL, device=device)
y_seq = torch.randint(0, 4, (800,), device=device)
seq_ds  = TensorDataset(X_seq[:640], y_seq[:640])
seq_val = TensorDataset(X_seq[640:], y_seq[640:])
seq_trl = DataLoader(seq_ds,  batch_size=32, shuffle=True)
seq_vll = DataLoader(seq_val, batch_size=64)

for name, Block in [('post-norm', TransformerBlockPostNorm),
                     ('pre-norm',  TransformerBlockPreNorm)]:
    block = Block().to(device)
    head  = nn.Linear(D_MODEL, 4).to(device)
    params = list(block.parameters()) + list(head.parameters())
    opt   = torch.optim.Adam(params, lr=1e-3)
    for _ in range(30):
        block.train(); head.train()
        for xb, yb in seq_trl:
            opt.zero_grad()
            out = head(block(xb).squeeze(1))
            nn.CrossEntropyLoss()(out, yb).backward()
            opt.step()
    block.eval(); head.eval()
    with torch.no_grad():
        correct = sum(
            (head(block(xb).squeeze(1)).argmax(1)==yb).sum().item()
            for xb, yb in seq_vll
        )
    print(f'{name}  val accuracy: {correct/len(seq_val):.4f}')

print('Pre-norm typically converges more stably; enables training '
      'deeper networks without LR warmup.')


## Real-World Example 2: RMSNorm — LLaMA-Style Implementation

In [ ]:
# RMSNorm used in LLaMA, Mistral, Gemma: removes mean-centering step.
# Speed advantage: ~7-15% faster than LayerNorm on large d_model.
# Hypothesis: the mean shift matters less than the scale normalisation.

import time

torch.manual_seed(20)


class RMSNormLLaMA(nn.Module):
    '''Production RMSNorm as used in LLaMA/Mistral implementations.
    Normalises by RMS of activations; no mean subtraction.
    '''
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps   = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def _norm(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x):
        return self.weight * self._norm(x.float()).type_as(x)


# Benchmark: RMSNorm vs LayerNorm throughput on large activation tensors
BENCH_BATCH = 256
SEQ_LEN     = 512
D_LARGE     = 4096   # LLaMA-7B hidden size

rms_norm = RMSNormLLaMA(D_LARGE).to(device)
lay_norm = nn.LayerNorm(D_LARGE).to(device)

# Create a small test tensor (avoid OOM on CPU with large dims)
D_TEST = min(D_LARGE, 512)   # cap at 512 for CPU benchmark
rms_norm_t = RMSNormLLaMA(D_TEST).to(device)
lay_norm_t = nn.LayerNorm(D_TEST).to(device)
x_bench    = torch.randn(64, 128, D_TEST, device=device)

N_ITERS = 200
t0 = time.perf_counter()
for _ in range(N_ITERS):
    with torch.no_grad(): _ = rms_norm_t(x_bench)
t_rms = time.perf_counter() - t0

t0 = time.perf_counter()
for _ in range(N_ITERS):
    with torch.no_grad(): _ = lay_norm_t(x_bench)
t_ln = time.perf_counter() - t0

print(f'RMSNorm time ({N_ITERS} iters): {t_rms*1000:.1f} ms')
print(f'LayerNorm time ({N_ITERS} iters): {t_ln*1000:.1f} ms')
speedup = t_ln / t_rms
print(f'RMSNorm speedup: {speedup:.2f}x  '
      f'({'faster' if speedup>1 else 'slower'} than LayerNorm)')
print('On GPU (fp16), RMSNorm is typically 7-15% faster than LayerNorm.')
print('Used in: LLaMA, LLaMA-2, Mistral, Gemma, Falcon.')


## Real-World Example 3: Gradient Flow — Pre-Norm vs Post-Norm Analysis

In [ ]:
# Measure gradient magnitudes at each layer for pre-norm vs post-norm.
# Post-Norm: gradients shrink geometrically with depth (vanishing gradient).
# Pre-Norm: gradients remain roughly constant — key advantage for deep networks.

torch.manual_seed(30)
N_LAYERS = 8
D_GF     = 32


class DeepPostNorm(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(D_GF, D_GF) for _ in range(N_LAYERS)
        ])
        self.norms  = nn.ModuleList([
            nn.LayerNorm(D_GF) for _ in range(N_LAYERS)
        ])

    def forward(self, x):
        for lin, norm in zip(self.layers, self.norms):
            x = norm(x + torch.relu(lin(x)))   # post-norm
        return x.mean(dim=-1)


class DeepPreNorm(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(D_GF, D_GF) for _ in range(N_LAYERS)
        ])
        self.norms  = nn.ModuleList([
            nn.LayerNorm(D_GF) for _ in range(N_LAYERS)
        ])

    def forward(self, x):
        for lin, norm in zip(self.layers, self.norms):
            x = x + torch.relu(lin(norm(x)))   # pre-norm
        return x.mean(dim=-1)


def measure_grad_norms(ModelClass, x_in, y_in):
    '''Compute per-layer gradient L2 norm after one backward pass.'''
    model = ModelClass().to(device)
    out   = model(x_in)
    nn.MSELoss()(out, y_in.float()).backward()
    norms = []
    for layer_lin in model.layers:
        g = layer_lin.weight.grad
        if g is not None:
            norms.append(g.norm().item())
    return norms


x_gf = torch.randn(128, D_GF, device=device)
y_gf = torch.randint(0, 2, (128,), device=device)

post_norms = measure_grad_norms(DeepPostNorm, x_gf, y_gf)
pre_norms  = measure_grad_norms(DeepPreNorm,  x_gf, y_gf)

print(f'{'Layer':<8} {'Post-Norm grad':>16} {'Pre-Norm grad':>14}')
print('-' * 42)
for i, (p, q) in enumerate(zip(post_norms, pre_norms)):
    print(f'{i+1:<8} {p:>16.6f} {q:>14.6f}')

ratio_post = max(post_norms) / (min(post_norms) + 1e-12)
ratio_pre  = max(pre_norms)  / (min(pre_norms)  + 1e-12)
print(f'\nPost-Norm gradient ratio (max/min): {ratio_post:.1f}x')
print(f'Pre-Norm  gradient ratio (max/min): {ratio_pre:.1f}x')
print('Smaller ratio = more uniform gradient flow = easier to train deep networks.')
